# Tomato Plant Disease Detection
**Computer Vision + Machine Learning Pipeline**

This notebook walks through every stage of the system interactively:

| # | Stage | Module |
|---|-------|--------|
| 1 | Dataset loading & EDA | `src/preprocessing.py` |
| 2 | Image preprocessing & colour-space conversion | `src/preprocessing.py` |
| 3 | Feature extraction (colour + texture) | `src/feature_extraction.py` |
| 4 | PCA dimensionality reduction | `src/dimensionality_reduction.py` |
| 5 | Model training — KNN, SVM, Random Forest | `src/models.py` |
| 6 | Evaluation — metrics, confusion matrix, ROC | `src/evaluation.py` |
| 7 | Disease prediction on a single image | `src/predictor.py` |

## 0 · Setup

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')

# Make sure the src/ package is importable when running from the notebook directory
if '.' not in sys.path:
    sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 12

print('All imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIR       = 'data/raw'        # root of PlantVillage tomato sub-folders
RESULTS_DIR    = 'results'
MODELS_DIR     = 'models'
MAX_PER_CLASS  = 200               # set to None to use the full dataset
IMAGE_SIZE     = (128, 128)        # (H, W)
HIST_BINS      = 32
PCA_VARIANCE   = 0.95              # retain 95 % of variance
RANDOM_STATE   = 42
TEST_SIZE      = 0.20
VAL_SIZE       = 0.10

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
print(f'Data dir   : {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')

---
## 1 · Dataset Loading & Exploratory Data Analysis

In [ ]:
from src.preprocessing import (
    load_dataset, split_data, get_class_distribution,
    TOMATO_CLASSES, LABEL_TO_CLASS
)

images, labels, found_classes = load_dataset(
    data_dir      = DATA_DIR,
    image_size    = IMAGE_SIZE,
    max_per_class = MAX_PER_CLASS,
)

print(f'\nImages shape : {images.shape}   dtype: {images.dtype}')
print(f'Labels shape : {labels.shape}')
print(f'Classes found: {len(found_classes)}')

In [ ]:
# ── Re-map labels to contiguous 0-based range ──────────────────────────────
unique_labels        = sorted(np.unique(labels))
remap                = {old: new for new, old in enumerate(unique_labels)}
labels               = np.array([remap[l] for l in labels], dtype=np.int32)
class_names_ordered  = [LABEL_TO_CLASS[old] for old in unique_labels]
n_classes            = len(class_names_ordered)

short_names = [c.replace('Tomato__', '').replace('_', ' ')
               for c in class_names_ordered]

print('Class mapping:')
for i, name in enumerate(class_names_ordered):
    print(f'  {i:2d}  {name}')

In [ ]:
# ── Class distribution bar chart ───────────────────────────────────────────
dist   = get_class_distribution(labels, class_names_ordered)
counts = [dist.get(n, 0) for n in class_names_ordered]

fig, ax = plt.subplots(figsize=(13, 4))
colors  = plt.cm.tab10(np.linspace(0, 1, n_classes))
bars    = ax.bar(short_names, counts, color=colors, edgecolor='white')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(cnt), ha='center', va='bottom', fontsize=8)
ax.set_xticklabels(short_names, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Image Count')
ax.set_title('Class Distribution — Tomato Leaf Dataset')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample images grid ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(min(n_classes, 10)):
    idx = np.where(labels == i)[0][0]
    axes[i].imshow(images[idx])
    axes[i].set_title(short_names[i], fontsize=8)
    axes[i].axis('off')

# Hide unused axes
for j in range(n_classes, len(axes)):
    axes[j].axis('off')

fig.suptitle('One Sample per Class', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2 · Preprocessing — Colour-Space Conversion

In [ ]:
from src.preprocessing import to_hsv, to_lab, to_grayscale

# Pick one sample image for visualisation
sample = images[np.where(labels == 0)[0][0]]   # first healthy leaf

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

conversions = [
    ('RGB (original)', sample),
    ('HSV',           to_hsv(sample)),
    ('LAB',           to_lab(sample)),
    ('Grayscale',     to_grayscale(sample)),
]
channel_labels = [
    ['R', 'G', 'B'],
    ['H', 'S', 'V'],
    ['L', 'A', 'B'],
    ['Gray'],
]

for col, (title, img) in enumerate(conversions):
    # Top row: rendered image
    ax = axes[0, col]
    if img.ndim == 2:
        ax.imshow(img, cmap='gray')
    else:
        display = np.clip(img, 0, 1)
        ax.imshow(display)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

    # Bottom row: channel histograms
    ax2 = axes[1, col]
    chs = [img] if img.ndim == 2 else [img[:,:,c] for c in range(img.shape[2])]
    colors_ch = ['red','green','blue'] if title.startswith('RGB') else \
                ['gold','cornflowerblue','tomato']
    for c_idx, ch in enumerate(chs):
        hist, edges = np.histogram(ch.ravel(), bins=64, range=(0,1))
        ax2.plot(edges[:-1], hist,
                 color=colors_ch[c_idx % len(colors_ch)],
                 label=channel_labels[col][c_idx] if c_idx < len(channel_labels[col]) else '',
                 alpha=0.8, linewidth=1.5)
    ax2.set_xlabel('Pixel Value', fontsize=8)
    ax2.set_ylabel('Count', fontsize=8)
    ax2.legend(fontsize=7)
    ax2.grid(alpha=0.3)

fig.suptitle('Colour-Space Conversions & Channel Histograms', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/colour_space_conversions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Train / Val / Test split ───────────────────────────────────────────────
(X_train, X_val, X_test,
 y_train, y_val, y_test) = split_data(
    images, labels,
    test_size    = TEST_SIZE,
    val_size     = VAL_SIZE,
    random_state = RANDOM_STATE,
)

print(f'Train : {X_train.shape}  labels: {y_train.shape}')
print(f'Val   : {X_val.shape}  labels: {y_val.shape}')
print(f'Test  : {X_test.shape}  labels: {y_test.shape}')

---
## 3 · Feature Extraction

In [ ]:
from src.feature_extraction import FeatureExtractor

extractor = FeatureExtractor(
    use_color_histogram  = True,
    histogram_bins       = HIST_BINS,
    histogram_spaces     = ('rgb', 'hsv', 'lab'),
    use_color_statistics = True,
    use_glcm             = True,
    use_lbp              = True,
    use_hu_moments       = True,
)

print('Feature group sizes:')
total = 0
for group, size in extractor.feature_group_sizes().items():
    print(f'  {group:<25} {size:>5} dims')
    total += size
print(f'  {"TOTAL":<25} {total:>5} dims')

In [ ]:
print('Extracting training features ...')
F_train = extractor.extract_batch(X_train, desc='Train')
print('Extracting validation features ...')
F_val   = extractor.extract_batch(X_val,   desc='Val')
print('Extracting test features ...')
F_test  = extractor.extract_batch(X_test,  desc='Test')

print(f'\nFeature matrix shapes:')
print(f'  F_train : {F_train.shape}')
print(f'  F_val   : {F_val.shape}')
print(f'  F_test  : {F_test.shape}')

In [ ]:
# ── Visualise feature distributions per class (first 6 features) ──────────
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()
cmap = plt.cm.tab10

for feat_idx in range(6):
    ax = axes[feat_idx]
    for cls_idx in range(n_classes):
        mask = y_train == cls_idx
        vals = F_train[mask, feat_idx]
        ax.hist(vals, bins=30, alpha=0.4,
                color=cmap(cls_idx / max(n_classes-1,1)),
                label=short_names[cls_idx], density=True)
    ax.set_title(f'Feature dim {feat_idx}', fontsize=9)
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    if feat_idx == 0:
        ax.legend(fontsize=6, ncol=2)

fig.suptitle('Distribution of First 6 Features per Class', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 · PCA Dimensionality Reduction

In [ ]:
from src.dimensionality_reduction import PCAReducer, select_n_components

# Survey how many components are needed at different variance levels
print('Components needed at various variance thresholds:')
thresh_map = select_n_components(F_train, [0.80, 0.90, 0.95, 0.99])

In [ ]:
reducer = PCAReducer(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)

P_train = reducer.fit_transform(F_train)
P_val   = reducer.transform(F_val)
P_test  = reducer.transform(F_test)

print(f'\nReduced dimensions: {F_train.shape[1]}  →  {P_train.shape[1]}')

In [ ]:
# ── Scree plot + cumulative variance ──────────────────────────────────────
fig = reducer.plot_explained_variance(
    save_path = f'{RESULTS_DIR}/pca_explained_variance.png',
    threshold = PCA_VARIANCE,
)
plt.show()

In [ ]:
# ── 2-D PCA projection coloured by class ─────────────────────────────────
fig = reducer.plot_2d_projection(
    F_train, y_train,
    class_names = class_names_ordered,
    save_path   = f'{RESULTS_DIR}/pca_2d_projection.png',
)
plt.show()

In [ ]:
reducer.save(f'{MODELS_DIR}/pca_reducer.pkl')

---
## 5 · Model Training — KNN · SVM · Random Forest

In [ ]:
from src.models import build_all_models, train_all_models

models = build_all_models()

# Set USE_GRID_SEARCH = True for hyperparameter tuning (takes longer)
USE_GRID_SEARCH = False

models = train_all_models(
    models, P_train, y_train,
    use_grid_search = USE_GRID_SEARCH,
)

In [ ]:
# Persist all models
for key, wrapper in models.items():
    wrapper.save(MODELS_DIR)

---
## 6 · Evaluation

In [ ]:
from src.evaluation import (
    full_evaluation, compare_models,
    plot_model_comparison, plot_training_times,
    plot_confusion_matrix, plot_roc_curves,
)

all_results = []

for key, wrapper in models.items():
    print(f'\n{"─"*55}')
    print(f'  Evaluating: {wrapper.name}')
    result = full_evaluation(
        wrapper, P_test, y_test,
        class_names = class_names_ordered,
        results_dir = f'{RESULTS_DIR}/test',
    )
    all_results.append(result)
    plt.close('all')

In [ ]:
# ── Summary comparison table ───────────────────────────────────────────────
df_compare = compare_models(
    all_results,
    save_path = f'{RESULTS_DIR}/model_comparison.csv',
)
df_compare.style.highlight_max(axis=0, color='#d4f0c0').format('{:.4f}')

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────
fig = plot_model_comparison(
    df_compare,
    save_path = f'{RESULTS_DIR}/model_comparison.png',
)
plt.show()

In [ ]:
# ── Training time chart ────────────────────────────────────────────────────
fig = plot_training_times(
    models,
    save_path = f'{RESULTS_DIR}/training_times.png',
)
plt.show()

### 6a · Confusion Matrices

In [ ]:
for key, wrapper in models.items():
    y_pred = wrapper.predict(P_test)
    fig = plot_confusion_matrix(
        y_test, y_pred,
        class_names = class_names_ordered,
        model_name  = wrapper.name,
        normalize   = True,
        save_path   = f'{RESULTS_DIR}/test/{wrapper.name.lower().replace(" ","_")}_confusion.png',
    )
    plt.show()
    plt.close('all')

### 6b · ROC Curves

In [ ]:
for key, wrapper in models.items():
    try:
        y_proba = wrapper.predict_proba(P_test)
        fig = plot_roc_curves(
            y_test, y_proba,
            class_names = class_names_ordered,
            model_name  = wrapper.name,
            save_path   = f'{RESULTS_DIR}/test/{wrapper.name.lower().replace(" ","_")}_roc.png',
        )
        plt.show()
        plt.close('all')
    except Exception as e:
        print(f'  Skipping ROC for {wrapper.name}: {e}')

### 6c · Per-class F1 Heatmap

In [ ]:
from sklearn.metrics import f1_score

f1_data = {}
for key, wrapper in models.items():
    y_pred = wrapper.predict(P_test)
    f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0)
    f1_data[wrapper.name] = f1_per_class

df_f1 = pd.DataFrame(f1_data, index=short_names)

fig, ax = plt.subplots(figsize=(8, max(5, n_classes * 0.5)))
sns.heatmap(
    df_f1, annot=True, fmt='.2f', cmap='YlOrRd',
    vmin=0, vmax=1, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'F1 Score'},
)
ax.set_title('Per-Class F1 Score — All Models', fontsize=12)
ax.set_xlabel('Model', fontsize=10)
ax.set_ylabel('Disease Class', fontsize=10)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7 · Disease Prediction — Single Image

In [ ]:
from src.predictor import DiseasePredictor, save_predictor, load_predictor

# Pick the best model by weighted F1
best_model_key = df_compare['f1_weighted'].idxmax()
print(f'Best model: {best_model_key}')

predictor = DiseasePredictor(
    model       = models[best_model_key],
    extractor   = extractor,
    reducer     = reducer,
    class_names = class_names_ordered,
    image_size  = IMAGE_SIZE,
)

save_predictor(predictor, f'{MODELS_DIR}/best_predictor.pkl')

In [ ]:
# ── Predict on a random test image ────────────────────────────────────────
sample_idx  = np.random.randint(len(X_test))
sample_img  = X_test[sample_idx]
true_label  = y_test[sample_idx]
true_class  = class_names_ordered[true_label]

result = predictor.predict_image(sample_img)
top3   = predictor.predict_top_k(sample_img, k=3)

print(predictor.format_prediction(result, top3))
print(f'\nGround truth: {true_class}')
print(f'Correct     : {result["predicted_label"] == true_label}')

In [ ]:
# ── Visualise prediction with confidence bar ───────────────────────────────
fig = plt.figure(figsize=(14, 4))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2], figure=fig)

# Left: image
ax_img = fig.add_subplot(gs[0])
ax_img.imshow(np.clip(sample_img, 0, 1))
correct = result['predicted_label'] == true_label
border_color = 'green' if correct else 'red'
for spine in ax_img.spines.values():
    spine.set_edgecolor(border_color)
    spine.set_linewidth(3)
true_short = true_class.replace('Tomato__', '').replace('_', ' ')
pred_short = result['predicted_class'].replace('Tomato__', '').replace('_', ' ')
ax_img.set_title(
    f'True: {true_short}\nPred: {pred_short}  '
    f'({"✓" if correct else "✗"})',
    fontsize=9, color=border_color
)
ax_img.axis('off')

# Right: top-k confidence bars
ax_bar = fig.add_subplot(gs[1])
probs_sorted = sorted(result['probabilities'].items(),
                      key=lambda x: x[1], reverse=True)[:5]
names_bar  = [k.replace('Tomato__', '').replace('_', ' ') for k, _ in probs_sorted]
values_bar = [v for _, v in probs_sorted]
bar_colors = ['#4CAF50' if n == pred_short else '#90CAF9' for n in names_bar]
ax_bar.barh(names_bar[::-1], values_bar[::-1],
            color=bar_colors[::-1], edgecolor='white')
ax_bar.set_xlim(0, 1)
ax_bar.set_xlabel('Confidence')
ax_bar.set_title('Top-5 Class Probabilities', fontsize=10)
for i, v in enumerate(values_bar[::-1]):
    ax_bar.text(v + 0.01, i, f'{v*100:.1f}%', va='center', fontsize=9)
ax_bar.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/prediction_example.png', dpi=150, bbox_inches='tight')
plt.show()

### Predict on a custom image file

In [ ]:
# Set IMAGE_PATH to any tomato leaf image on your machine
IMAGE_PATH = 'data/raw/Tomato__healthy/0a0d9e2e-9df6-4ef4-85da-bf853f25d32f___RS_HL 0064.JPG'

if Path(IMAGE_PATH).exists():
    result = predictor.predict_image(IMAGE_PATH)
    top3   = predictor.predict_top_k(IMAGE_PATH, k=3)
    print(predictor.format_prediction(result, top3))

    import cv2
    raw = cv2.imread(IMAGE_PATH)
    raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 4))
    plt.imshow(raw)
    plt.title(result['predicted_class'].replace('Tomato__','').replace('_',' '))
    plt.axis('off')
    plt.show()
else:
    print(f'File not found: {IMAGE_PATH}')
    print('Update IMAGE_PATH to a valid tomato leaf image.')

---
## 8 · Batch Prediction Demo

In [ ]:
# Predict on 10 random test images and show a summary grid
n_demo  = 10
indices = np.random.choice(len(X_test), n_demo, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.ravel()

for i, idx in enumerate(indices):
    img    = X_test[idx]
    true_l = y_test[idx]
    res    = predictor.predict_image(img)

    correct = res['predicted_label'] == true_l
    color   = 'green' if correct else 'red'
    symbol  = '✓' if correct else '✗'

    axes[i].imshow(np.clip(img, 0, 1))
    pred_s = res['predicted_class'].replace('Tomato__','').replace('_',' ')
    true_s = class_names_ordered[true_l].replace('Tomato__','').replace('_',' ')
    axes[i].set_title(
        f'T: {true_s[:18]}\nP: {pred_s[:18]} {symbol}\n{res["confidence"]*100:.0f}%',
        fontsize=7, color=color
    )
    axes[i].axis('off')
    for spine in axes[i].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

fig.suptitle(f'Batch Prediction ({best_model_key}) — Green=Correct, Red=Wrong',
             fontsize=11)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/batch_prediction_demo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9 · Summary

In [ ]:
print('\n' + '█'*60)
print('  FINAL RESULTS SUMMARY')
print('█'*60)
print(f'  Dataset          : {len(images)} images, {n_classes} classes')
print(f'  Feature dims     : {F_train.shape[1]}  →  {P_train.shape[1]} after PCA')
print(f'  PCA variance     : {reducer.explained_variance_ratio.sum()*100:.2f}%')
print()
print(df_compare.to_string(float_format='{:.4f}'.format))
print()
best = df_compare['f1_weighted'].idxmax()
print(f'  Best model       : {best}')
print(f'  Best F1-weighted : {df_compare.loc[best, "f1_weighted"]:.4f}')
print(f'  Best Accuracy    : {df_compare.loc[best, "accuracy"]:.4f}')
print()
print(f'  Saved artefacts:')
for f in sorted(Path(RESULTS_DIR).rglob('*.png')):
    print(f'    {f}')
for f in sorted(Path(MODELS_DIR).rglob('*')):
    print(f'    {f}')
print('█'*60)